### Building a lookup and join pipeline: 45 messy CSV filenames need to be matched to 45 clean rows in stations_master.csv, then each raw file's pollution data gets summarized into one row of statistics per station. Think of it like a filing clerk matching loose paper folders (filenames) to labeled drawers (the master table) using a name-matching rule, then reading each folder and writing a summary card.

- Load — stations_api.csv gets read into a DataFrame called df (the raw, messy, 315-row version).

- Extract & shape — You pull out just the columns you need, clean up the names, and build a new DataFrame called stations (which eventually becomes master) — down to 45 unique, tidy rows.

- Save — master gets written back out to disk as stations_master.csv, so it's now a permanent file you (and your next scripts) can reload anytime without repeating steps 1–2.

In [25]:
import pandas as pd
import re
from pathlib import Path

API_PATH = Path("../data/api/stations_api.csv")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(API_PATH)
stations = df[['station','latitude','longitude']].drop_duplicates().reset_index(drop=True)

stations['station_clean'] = stations['station'].str.strip()
stations['network'] = stations['station_clean'].str.extract(r'-\s*(\w+)\s*$')[0]
stations['station_name'] = stations['station_clean'].str.replace(
    r',?\s*Delhi\s*-\s*\w+\s*$', '', regex=True
).str.strip()

dup_names = stations['station_name'].value_counts()
dup_names = dup_names[dup_names > 1].index.tolist()

def make_unique_name(row):
    if row['station_name'] in dup_names:
        return f"{row['station_name']} ({row['network']})"
    return row['station_name']

stations['station_name_unique'] = stations.apply(make_unique_name, axis=1)
stations = stations.sort_values('station_name_unique').reset_index(drop=True)
stations['station_id'] = ['STN_' + str(i+1).zfill(3) for i in range(len(stations))]

master = stations[['station_id','station_name_unique','network','latitude','longitude']].rename(
    columns={'station_name_unique': 'station_name'}
)
master['latitude'] = master['latitude'].round(6)
master['longitude'] = master['longitude'].round(6)

master.to_csv(OUT_DIR / "stations_master.csv", index=False)
print("Saved:", OUT_DIR / "stations_master.csv")
master.head()

Saved: ../data/processed/stations_master.csv


,station_id,station_name,network,latitude,longitude
0,STN_001,Alipur,DPCC,28.815329,77.153010
1,STN_002,Anand Vihar,DPCC,28.647622,77.315809
2,STN_003,Ashok Vihar,DPCC,28.695381,77.181665
3,STN_004,Aya Nagar,IITM,28.470691,77.109936
4,STN_005,Bawana,DPCC,28.776200,77.051074


In [14]:
MASTER_PATH = Path("../data/processed/stations_master.csv")

In [18]:
df["station"]

0                   DTU, Delhi - CPCB
1                   DTU, Delhi - CPCB
2             IIT Delhi, Delhi - IITM
3             IIT Delhi, Delhi - IITM
4             R K Puram, Delhi - DPCC
                    ...              
310    IGI Airport (T3), Delhi - IITM
311            Sirifort, Delhi - CPCB
312            Sirifort, Delhi - CPCB
313            Sirifort, Delhi - CPCB
314            Sirifort, Delhi - CPCB
Name: station, Length: 315, dtype: object

In [21]:
df[['station','latitude','longitude']]

,station,latitude,longitude
0,"DTU, Delhi - CPCB",28.750050,77.111261
1,"DTU, Delhi - CPCB",28.750050,77.111261
2,"IIT Delhi, Delhi - IITM",28.542460,77.191651
3,"IIT Delhi, Delhi - IITM",28.542460,77.191651
4,"R K Puram, Delhi - DPCC",28.563262,77.186937
...,...,...,...
310,"IGI Airport (T3), Delhi - IITM",28.562776,77.118005
311,"Sirifort, Delhi - CPCB",28.550425,77.215938
312,"Sirifort, Delhi - CPCB",28.550425,77.215938
313,"Sirifort, Delhi - CPCB",28.550425,77.215938


In [22]:
df.head()

,country,state,city,station,last_update,latitude,longitude,pollutant_id,pollutant_min,pollutant_max,pollutant_avg
0,India,Delhi,Delhi,"DTU, Delhi - CPCB",11-07-2026 18:00:00,28.750050,77.111261,PM10,28.0,301.0,139.0
1,India,Delhi,Delhi,"DTU, Delhi - CPCB",11-07-2026 18:00:00,28.750050,77.111261,NH3,7.0,11.0,9.0
2,India,Delhi,Delhi,"IIT Delhi, Delhi - IITM",11-07-2026 18:00:00,28.542460,77.191651,PM2.5,24.0,179.0,66.0
3,India,Delhi,Delhi,"IIT Delhi, Delhi - IITM",11-07-2026 18:00:00,28.542460,77.191651,CO,15.0,21.0,19.0
4,India,Delhi,Delhi,"R K Puram, Delhi - DPCC",11-07-2026 18:00:00,28.563262,77.186937,PM2.5,5.0,143.0,46.0


In [24]:
master

,station_id,station_name,network,latitude,longitude
0,STN_001,Alipur,DPCC,28.815329,77.153010
1,STN_002,Anand Vihar,DPCC,28.647622,77.315809
2,STN_003,Ashok Vihar,DPCC,28.695381,77.181665
3,STN_004,Aya Nagar,IITM,28.470691,77.109936
4,STN_005,Bawana,DPCC,28.776200,77.051074
5,STN_006,Burari Crossing,IITM,28.725650,77.201157
6,STN_007,CRRI Mathura Road,IITM,28.551200,77.273574
7,STN_008,Cantonment Area,DPCC,28.594169,77.125100
8,STN_009,Chandni Chowk,IITM,28.656756,77.227234
9,STN_010,Commonwealth Sports Complex,DPCC,28.615828,77.271992
